<a href="https://colab.research.google.com/github/SaraAlinejad/vae_test_1/blob/main/Alinejad_Assignment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Chemical Applications of Machine Learning (CHEM 4930/5610) - Spring 2026

### Assignment 3 - Deadline 2/27/2026
Points 10

#### General Comments
All figures and graph should have approriate labels on the two axis, and should include a legend with appropriate labels of the different plots.

The notebook should be return in working format. That is, I should be able to reset all the output and re-run all the cells and get the same results as you obtained.

**You should start by saving a copy of the notebook to your Google Drive so you preserve all changes.**

**Please add your name as a suffix to the filname**

**Student Name**: Sara Alinejad

**AI usage statement:**
I used AI to check whether I missed a step all the way and tidy up my code. I learnt usefull spot checking and verification tips. for quickly fixing  errors I got with the rf pipline AI was used.   

### Task 1 - 10 points

In this task, we will consider data from this paper:
- Enhancing Permeability Prediction of Heterobifunctional Degraders Using Machine Learning and Metadynamics-Informed 3D Molecular Descriptors - [DOI:10.1021/acs.jcim.5c01600](https://doi.org/10.1021/acs.jcim.5c01600)

Where the authors consider the Permeability of so-called PROTAC compounds that are large and flexible molecules used in Targeted Protein Degradation.

All the dataset used in the paper, and the code use to obtain the results are given in this following Github repository:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD  

The specfic dataset that we use 32 PROTAC compounds with measured passive permeability (given in nm/s) and includes 17 2D features calculated by RDKit (see [here](https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/data/calculate_2d_properties.py) for the script they are calculated), and 3 3D "ensemble" features that are obtained from molecular dynamics simulations as described in the paper.

The target value is the measured passive permeability that is experimentaly measured and given in nm/s.

The dataset can be seen here:
- https://github.com/brykimjh/degrader-permeability-ml3d-metaD/blob/main/outputs/ml_models/model_data.csv

Where the log10 transformed passive permeability is given by `P_appLog`. Note that here the passive permeability has been log transformed in based 10 using `np.log10`.

The 2D features obtained using RDKit are:
```
[
 'Molecular Weight (MW)', 'CharVol (characteristic volume)',
 'Flexibility (number of rotatable bonds / number of bonds)',
 'Number of Heavy Atoms (HA)', 'RingAtoms', 'Halogens', 'HeteroAtoms',
 'RotBonds (NRotB)', 'AllBonds', 'RingCount', 'NumStereo',
 'Fraction of sp3 Carbon Atoms (FSP3)', 'Hydrogen Bond Donors (HBD)',
 'Hydrogen Bond Acceptors (HBA)', 'cLogD^7.4',
 'Topological polar surface area (TPSA)',
 'Total non-polar surface area (TNSA)'
]
```
that includes various standard descriptors/features implemented in RDKit

and the 3D "ensemble" features obtained using MD simulations are
```
[
 'Ensemble_Average_PSA_Chloroform_ANI',
 'Ensemble_Average_Num_IMHB_Chloroform_ANI',
 'Ensemble_Average_RadiusOfGyration_Chloroform_ANI'
]
```
which inludes the average polar surface area (PSA), the average number of intramolecular hydrogen bonds (IMHB), and the average radius of gyration (that measures if the molecule is compact or extended).

#### A)
Split the dataset into compounds with two classes with low and high permeability evenly so each class has 16 compounds.

#### B)
For all 32 compounds, visualize the 2D chemical structure using RDKit and the mols2grid package, and show their measured passive permeability in nm/s and their low/high permeability classifciation.

#### C)
Perform classifcation using random forest classification model using the following hyperparameters:
- Number of tree: 100
- Maximum depth of each tree: 2
- Maximum number of features for each tree: 50% of the number of input features (rounded up if that number is a fraction)

You should perform the classification using 3 different feature set:
- 2D RDKit features
- 3D Ensemble features
- Combined set of 2D and 3D features

For each case, perform cross validation (CV) using a CV strategy of your choice, and obtain the average and standard deviation of metrics that measure the performance, including:
- Accuracy
- Precision
- Receiver Operating Characteristic Area Under the Curve (ROC AUC)  

In your analysis and metrics, the high permeability class should be consider as the postive class.

Based on your analysis, which feature set gives the best results for the classifcation?

#### D) - Optional for 2 points
Repeat your analysis from C) using a $k$ nearest neighbors classifer (i.e., $k$ nearest neighbors vote). Use the default value of 5 neighbors.

Do you obtain the same results as in C)?



In [ ]:
%%bash
# Download the main model data and the molecular structure data
BASE_URL="https://raw.githubusercontent.com/brykimjh/degrader-permeability-ml3d-metaD/main"
rm -f model_data.csv mol_data.csv

wget ${BASE_URL}/outputs/ml_models/model_data.csv &> /dev/null
wget ${BASE_URL}/data/mol_data.csv &> /dev/null

ls -l *.csv

In [ ]:
!pip install rdkit mols2grid lux-api -q # lux is useful visualization package for dataframes
import warnings
warnings.filterwarnings("ignore")
import math
import pandas as pd
import numpy as np
import lux  #on top of the dataframe, there will be a tab; Toggle Pandas/lux, click on it and all usufull plots can be seen
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import matplotlib.patches as mpatches
import seaborn  as sns
import mols2grid
from rdkit import Chem
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_validate, ShuffleSplit, StratifiedKFold
from sklearn.metrics import make_scorer, accuracy_score, precision_score, roc_auc_score
from sklearn.model_selection import StratifiedShuffleSplit


In [ ]:
df = pd.read_csv("model_data.csv")
print(f"Shape: {df.shape}")
df.head(3)

In [ ]:
#  A clean, style that looks better

PALETTE_LOW  = "#E05C5C"   # warm red  : low permeability
PALETTE_HIGH = "#4A90D9"   # steel blue : high permeability
PALETTE      = {"Low": PALETTE_LOW, "High": PALETTE_HIGH}

plt.rcParams.update({
    "figure.facecolor"     : "white",
    "axes.facecolor"       : "#F8F9FA",
    "axes.edgecolor"       : "#CCCCCC",
    "axes.grid"            : True,
    "grid.color"           : "white",
    "grid.linewidth"       : 1.2,
    "axes.spines.top"      : False,
    "axes.spines.right"    : False,
    "axes.labelsize"       : 12,
    "axes.titlesize"       : 13,
    "axes.titleweight"     : "bold",
    "xtick.labelsize"      : 10,
    "ytick.labelsize"      : 10,
    "legend.frameon"       : False,
    "legend.fontsize"      : 10,
    "font.family"          : "DejaVu Sans",
    "figure.dpi"           : 130,
})

In [ ]:
TARGET     = "P_appLog"   # log10-transformed passive permeability

FEATURES_2D = [
    "Molecular Weight (MW)",
    "CharVol (characteristic volume)",
    "Flexibility (number of rotatable bonds / number of bonds)",
    "Number of Heavy Atoms (HA)",
    "RingAtoms", "Halogens", "HeteroAtoms",
    "RotBonds (NRotB)", "AllBonds", "RingCount", "NumStereo",
    "Fraction of sp3 Carbon Atoms (FSP3)",
    "Hydrogen Bond Donors (HBD)",
    "Hydrogen Bond Acceptors (HBA)",
    "cLogD^7.4",
    "Topological polar surface area (TPSA)",
    "Total non-polar surface area (TNSA)",
]

FEATURES_3D = [
    "Ensemble_Average_PSA_Chloroform_ANI",
    "Ensemble_Average_Num_IMHB_Chloroform_ANI",
    "Ensemble_Average_RadiusOfGyration_Chloroform_ANI",
]

# Verifying all expected columns
missing = [c for c in FEATURES_2D + FEATURES_3D + [TARGET] if c not in df.columns]
if missing:
    print(" Missing columns:", missing)
else:
    print("All feature columns present.")

In [ ]:
#  Recover raw P_app from the log-transformed column
# The CSV stores log10(P_app); back-transform to nm/s for the threshold
if "P_app" not in df.columns:
    df["P_app"] = 10 ** df[TARGET]

# Compute median and assign labels
THRESHOLD = df["P_app"].median()   # nm/s

# Compounds strictly ABOVE the median :High; otherwise : Low
# Using strict inequality on an even-N dataset guarantees 16 / 16
df["Class"] = np.where(df["P_app"] > THRESHOLD, "High", "Low")

#  Sanity check
counts = df["Class"].value_counts()
print(f"Median P_app threshold : {THRESHOLD:.4f} nm/s")
print(f"Class counts           : {dict(counts)}")
assert counts["High"] == 16 and counts["Low"] == 16, "Split is not balanced!"
print(" Balanced split confirmed: 16 Low · 16 High")

In [ ]:
# Ranked bar chart — to show the split
plot_df = df.sort_values("P_app").reset_index(drop=True)
plot_df["rank"] = range(1, 33)

fig, ax = plt.subplots(figsize=(14, 5))
bar_colors = [PALETTE[c] for c in plot_df["Class"]]
ax.bar(plot_df["rank"], plot_df["P_app"],
       color=bar_colors, edgecolor="white", linewidth=0.6, width=0.85)

ax.axhline(THRESHOLD, color="#1a1a2e", lw=2, ls="--", zorder=5)
ax.text(33, THRESHOLD + 0.4,
        f" Median = {THRESHOLD:.1f} nm/s",
        va="bottom", ha="left", fontsize=9, color="#1a1a2e", style="italic")

ax.text(2, THRESHOLD * 0.38, "Low (n = 16)", ha="left",
        fontsize=11, color=PALETTE["Low"], fontweight="bold",
        zorder=20)

ax.text(18, THRESHOLD * 1.55, "High (n = 16)", ha="left",
        fontsize=11, color=PALETTE["High"], fontweight="bold",
        zorder=20)

ax.legend(
    handles=[mpatches.Patch(facecolor=PALETTE[c], label=c) for c in ["Low", "High"]],
    title="Class", loc="upper left"
)
ax.set_xlabel("Compound rank (ascending $P_{app}$)")
ax.set_ylabel(r"$P_{app}$ (nm s$^{-1}$)")
ax.set_title("Median Split into Low / High Permeability Classes\n"
             "32 Heterobifunctional Degraders")
ax.set_xlim(0.3, 32.7)
plt.tight_layout()
plt.show()

In [ ]:
feat_df = pd.read_csv("model_data.csv")   # descriptors + P_appLog
mol_df  = pd.read_csv("mol_data.csv")     # Smiles + P_app

# ── Check which P_app values are duplicated ───────────────────────────────────
print("Duplicate P_app values in mol_data.csv:")
print(mol_df[mol_df["P_app"].duplicated(keep=False)][["Compound","P_app"]])

In [ ]:
#Sort both files by P_appLog , same ordering

mol_df["_log_key"] = np.log10(mol_df["P_app"].round(1))
mol_sorted  = mol_df.sort_values("_log_key").reset_index(drop=True)
feat_sorted = feat_df.sort_values("P_appLog").reset_index(drop=True)

#  Positional concat
merged = pd.concat([mol_sorted, feat_sorted], axis=1)
merged = merged.loc[:, ~merged.columns.duplicated()]
merged = merged.drop(columns=["_log_key"])

print(f"Merged shape : {merged.shape}")   # (32, ...)

#  Verify
diff = np.abs(np.log10(merged["P_app"].round(1)) - merged["P_appLog"])
print(f"Max diff     : {diff.max():.8f}")

#  Spot-check
print(merged[merged["P_app"].round(1) == 3.2][["Compound", "P_app", "P_appLog"]])

In [ ]:
#  Median split
THRESHOLD = merged["P_app"].median()
merged["Class"]        = np.where(merged["P_app"] > THRESHOLD, "High", "Low")
merged["P_app (nm/s)"] = merged["P_app"].round(1)

print(f"Threshold : {THRESHOLD:.4f} nm/s")
print(f"Classes   : {dict(merged['Class'].value_counts())}")   #  16 / 16

In [ ]:
#Task 1B
mols2grid.display(
    pd.DataFrame(merged),
    smiles_col = "Smiles",
    subset     = ["img", "Compound", "P_app (nm/s)", "Class"],
    tooltip    = ["Compound", "P_app (nm/s)", "Class"],
    style      = {
        "Class": lambda x: (
            "background-color:#fde8e8; font-weight:bold; color:#c0392b;"
            if x == "Low" else
            "background-color:#e8f0fd; font-weight:bold; color:#2471a3;"
        )
    },
    n_cols = 4,
)

In [ ]:
# ── Feature matrices
X_2D  = merged[FEATURES_2D].values
X_3D  = merged[FEATURES_3D].values
X_all = np.hstack([X_2D, X_3D])

# ── Target: High = 1 (positive class), Low = 0
y = (merged["Class"] == "High").astype(int).values

print(f"X_2D  shape : {X_2D.shape}   : max_features = {math.ceil(0.5 * X_2D.shape[1])}")
print(f"X_3D  shape : {X_3D.shape}   : max_features = {math.ceil(0.5 * X_3D.shape[1])}")
print(f"X_all shape : {X_all.shape}  : max_features = {math.ceil(0.5 * X_all.shape[1])}")
print(f"\nTarget: {dict(zip(*np.unique(y, return_counts=True)))}  (1=High, 0=Low)")

In [ ]:
#  Stratified 5-fold CV
# Stratified = preserves 50/50 class ratio in each fold
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Scorers — High (1) is the positive class
SCORING = {
    "Accuracy" : make_scorer(accuracy_score),
    "Precision": make_scorer(precision_score, pos_label=1, zero_division=0),
    "ROC_AUC"  : make_scorer(roc_auc_score,   needs_proba=True),
}

print("CV strategy : StratifiedKFold(n_splits=5, shuffle=True)")
print("Positive class : High permeability (label = 1)")
print("Scorers :", list(SCORING.keys()))

In [ ]:
#  build RF with correct max_features for each feature set
def make_rf(n_features):
    return RandomForestClassifier(
        n_estimators = 100,
        max_depth    = 2,
        max_features = math.ceil(0.5 * n_features),   # 50% rounded up
        random_state = 42,
    )

# Run CV for each feature set
results = {}

for name, X in [("2D", X_2D), ("3D", X_3D), ("2D + 3D", X_all)]:
    clf    = make_rf(X.shape[1])
    cv_out = cross_validate(clf, X, y, cv=CV, scoring=SCORING, n_jobs=-1)

    results[name] = {
        metric: (cv_out[f"test_{metric}"].mean(),
                 cv_out[f"test_{metric}"].std())
        for metric in SCORING
    }
    print(f" {name} done")

In [ ]:
X_use = X_2D  # or X_3D / X_all

aucs = []
for k, (tr, te) in enumerate(CV.split(X_use, y), start=1):
    yte = y[te]
    uniq, cnt = np.unique(yte, return_counts=True)
    print(f"Fold {k} y_test counts:", dict(zip(uniq, cnt)))

    clf = make_rf(X_use.shape[1])
    clf.fit(X_use[tr], y[tr])

    proba = clf.predict_proba(X_use[te])[:, 1]
    auc = roc_auc_score(yte, proba)
    aucs.append(auc)
    print(f"Fold {k} AUC = {auc:.4f}")

print("AUC mean±std:", np.mean(aucs), np.std(aucs))

In [ ]:
SCORING = {
    "accuracy":  "accuracy",
    "precision": "precision",
    "roc_auc":   "roc_auc",
}

results = {}

for name, X in [("2D", X_2D), ("3D", X_3D), ("2D + 3D", X_all)]:
    clf = make_rf(X.shape[1])

    cv_out = cross_validate(
        clf, X, y,
        cv=CV,
        scoring=SCORING,
        n_jobs=-1,
        error_score="raise"
    )

    # store fold scores (arrays)
    results[name] = {
        "Accuracy":  cv_out["test_accuracy"],
        "Precision": cv_out["test_precision"],
        "ROC_AUC":   cv_out["test_roc_auc"],   # NOTE: test_roc_auc (lowercase!)
    }

    print(name, "AUC folds:", results[name]["ROC_AUC"])

In [ ]:
summary = {}
for fs in results:
    summary[fs] = {}
    for metric in ["Accuracy", "Precision", "ROC_AUC"]:
        vals = np.asarray(results[fs][metric], dtype=float)
        summary[fs][metric] = (vals.mean(), vals.std())

In [ ]:
metrics   = ["Accuracy", "Precision", "ROC_AUC"]
fs_names  = list(summary.keys())
fs_colors = ["#5B8DB8", "#E8A838", "#6CBB6C"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, metric in zip(axes, metrics):
    means = [summary[fs][metric][0] for fs in fs_names]
    stds  = [summary[fs][metric][1] for fs in fs_names]

    bars = ax.bar(
        fs_names, means, color=fs_colors,
        edgecolor="white", linewidth=0.8, width=0.55,
        yerr=stds, capsize=6
    )

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.02,
                f"{m:.2f}", ha="center", va="bottom",
                fontsize=9, fontweight="bold")

    ax.set_ylim(0, 1.18)
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("Feature set")

fig.suptitle("Task 1C — Random Forest | 5-fold Stratified CV | Error = ±1 SD",
             fontsize=12, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
rows = []
for fs in fs_names:
    row = {"Feature set": fs}
    for metric in ["Accuracy", "Precision", "ROC_AUC"]:
        mean, std = summary[fs][metric]
        row[metric] = f"{mean:.3f} ± {std:.3f}"
    rows.append(row)

results_table = pd.DataFrame(rows).set_index("Feature set")
display(pd.DataFrame(results_table))

In [ ]:
best = max(fs_names, key=lambda fs: summary[fs]["ROC_AUC"][0])

print("Ranking by ROC AUC (positive class = High) ")
for fs in sorted(fs_names, key=lambda fs: summary[fs]["ROC_AUC"][0], reverse=True):
    auc_m, auc_s = summary[fs]["ROC_AUC"]
    acc_m, _     = summary[fs]["Accuracy"]
    pre_m, _     = summary[fs]["Precision"]
    marker = " --> BEST" if fs == best else ""
    print(f"  {fs:10s}  ROC AUC={auc_m:.3f}±{auc_s:.3f}  "
          f"Acc={acc_m:.3f}  Prec={pre_m:.3f}{marker}")


  The 3D ensemble features (PSA, IMHB, Rgyr from WT-MetaD) outperform
  17 standard 2D RDKit descriptors despite being only 3 features.
  This matches the paper's finding: Rgyr alone has R²=0.50 with
  log(Papp), capturing the chameleonic compactness of PROTACs that
  no 2D topological descriptor can represent.

  Adding 2D on top of 3D (combined) may slightly hurt performance
  because the 14 weakly informative 2D features add noise which is a known
  issue with small datasets (n=32).

In [ ]:
clf = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
cross_validate(clf, X, y, cv=CV, scoring=SCORING, n_jobs=-1, error_score="raise")

In [ ]:
def mean_std(arr):
    return f"{np.mean(arr):.3f} ± {np.std(arr):.3f}"

print("Accuracy  :", mean_std(cv_out["test_accuracy"]))
print("Precision :", mean_std(cv_out["test_precision"]))
print("ROC AUC   :", mean_std(cv_out["test_roc_auc"]))

In [ ]:
results_knn = {}

for name, X in [("2D", X_2D), ("3D", X_3D), ("2D + 3D", X_all)]:
    clf = make_knn()

    cv_out = cross_validate(
        clf, X, y,
        cv=CV,
        scoring={"accuracy":"accuracy", "precision":"precision", "roc_auc":"roc_auc"},
        n_jobs=-1,
        error_score="raise"
    )

    results_knn[name] = {
        "Accuracy":  cv_out["test_accuracy"],
        "Precision": cv_out["test_precision"],
        "ROC_AUC":   cv_out["test_roc_auc"],
    }

In [ ]:
metrics   = ["Accuracy", "Precision", "ROC_AUC"]
fs_names  = list(results_knn.keys())
fs_colors = ["#5B8DB8", "#E8A838", "#6CBB6C"]

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))

for ax, metric in zip(axes, metrics):
    means = [results_knn[fs][metric].mean() for fs in fs_names]
    stds  = [results_knn[fs][metric].std()  for fs in fs_names]

    bars = ax.bar(fs_names, means, color=fs_colors,
                  edgecolor="white", linewidth=0.8, width=0.55,
                  yerr=stds, capsize=6)

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.02,
                f"{m:.2f}", ha="center", va="bottom",
                fontsize=9, fontweight="bold")

    ax.set_ylim(0, 1.18)
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_title(metric.replace("_", " "))
    ax.set_xlabel("Feature set")

fig.suptitle("Task 1D — kNN (k=5) | 5-fold Stratified CV | Error = ±1 SD",
             fontsize=12, fontweight="bold", y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
summary_knn = {}
for fs in results_knn:
    summary_knn[fs] = {}
    for metric in ["Accuracy", "Precision", "ROC_AUC"]:
        vals = np.asarray(results_knn[fs][metric], dtype=float)
        summary_knn[fs][metric] = (vals.mean(), vals.std())

In [ ]:
rows = []
for fs in ["2D", "3D", "2D + 3D"]:
    row = {"Feature set": fs}
    for metric in ["Accuracy", "Precision", "ROC_AUC"]:
        mean, std = summary_knn[fs][metric]
        row[metric] = f"{mean:.3f} ± {std:.3f}"
    rows.append(row)

display(pd.DataFrame(rows).set_index("Feature set"))

In [ ]:
#  RF vs KNN
metrics  = ["Accuracy", "Precision", "ROC_AUC"]
fs_names = ["2D", "3D", "2D + 3D"]
x = np.arange(len(fs_names))
w = 0.35   # bar width

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, metric in zip(axes, metrics):
    rf_means  = [summary[fs][metric][0]     for fs in fs_names]
    rf_stds   = [summary[fs][metric][1]     for fs in fs_names]
    knn_means = [summary_knn[fs][metric][0] for fs in fs_names]
    knn_stds  = [summary_knn[fs][metric][1] for fs in fs_names]

    b1 = ax.bar(x - w/2, rf_means,  w, label="RF",  color="#4A90D9",
                edgecolor="white", yerr=rf_stds,  capsize=5,
                error_kw=dict(elinewidth=1.4, ecolor="#333"))
    b2 = ax.bar(x + w/2, knn_means, w, label="KNN", color="#E05C5C",
                edgecolor="white", yerr=knn_stds, capsize=5,
                error_kw=dict(elinewidth=1.4, ecolor="#333"))

    # Annotate bar tops
    for bar, m, s in zip(b1, rf_means, rf_stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.02,
                f"{m:.2f}", ha="center", va="bottom", fontsize=8, color="#4A90D9")
    for bar, m, s in zip(b2, knn_means, knn_stds):
        ax.text(bar.get_x() + bar.get_width()/2, m + s + 0.02,
                f"{m:.2f}", ha="center", va="bottom", fontsize=8, color="#E05C5C")

    ax.set_xticks(x)
    ax.set_xticklabels(fs_names)
    ax.set_ylim(0, 1.22)
    ax.set_title(metric.replace("_", " "))
    ax.set_ylabel(metric.replace("_", " "))
    ax.set_xlabel("Feature set")
    ax.legend(loc="upper right", fontsize=9)

fig.suptitle("Task 1D — RF vs KNN  |  5-fold Stratified CV  |  error bars = ±1 SD\n"
             "Positive class: High permeability",
             fontsize=12, fontweight="bold", y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
best_rf  = max(fs_names, key=lambda fs: summary[fs]["ROC_AUC"][0])
best_knn = max(fs_names, key=lambda fs: summary_knn[fs]["ROC_AUC"][0])

print(" Best feature set by ROC AUC")
print(f"  RF --> '{best_rf}'   "
      f"AUC = {summary[best_rf]['ROC_AUC'][0]:.3f} ± {summary[best_rf]['ROC_AUC'][1]:.3f}")
print(f"  KNN --> '{best_knn}'  "
      f"AUC = {summary_knn[best_knn]['ROC_AUC'][0]:.3f} ± {summary_knn[best_knn]['ROC_AUC'][1]:.3f}")


the ranking of feature sets is the same (3D best), but RF is the more reliable classifier for this dataset (higher AUC, lower variance). RF is robust to feature scale and irrelevant features because it selects splits automatically (max_depth=2 acts as a strong regulariser) whereas KNN is more sensitive because it relies on Euclidean distance.

The large SD for both models reflects the small dataset (with 5-fold CV on n=32), each test fold has only ~ 6 samples, so individual fold scores are noisy